# 01 — Train Traffic Sign Classifier (GTSRB)

Trains a CNN to classify 43 German traffic sign classes.
This model is used as the second stage after sign detection (YOLOv8 finds the box → CNN classifies it).

**Prerequisites:** Run `python src/download_data.py --gtsrb` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from utils import PROJECT_ROOT, get_class_names, ensure_dirs

ensure_dirs()
DATA_DIR = PROJECT_ROOT / 'data' / 'gtsrb'
MODEL_PATH = PROJECT_ROOT / 'models' / 'sign_classifier.keras'

In [ ]:
from sign_classifier import load_gtsrb_data, build_model

(X_train, y_train), (X_test, y_test) = load_gtsrb_data(DATA_DIR)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Visualize samples
class_names = get_class_names('german')
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for ax in axes.flat:
    idx = np.random.randint(len(X_train))
    ax.imshow(X_train[idx])
    ax.set_title(class_names[np.argmax(y_train[idx])][:15], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
model = build_model(num_classes=43)
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=15, batch_size=64,
                    validation_data=(X_test, y_test))

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend()
plt.show()

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {acc:.4f}')

In [ ]:
model.save(str(MODEL_PATH))
print(f'Saved: {MODEL_PATH}')